In [27]:
import torch
import math

torch.set_printoptions(precision=2, sci_mode=False)

In [ ]:
# PyTorch Variant
# torch.nn.functional.scaled_dot_product_attention(Q, K, V)

In [28]:
def softmax_(x):
    if not isinstance(x, list):
        raise Exception("Please send a list for softmax")
    ess = [math.exp(xi) for xi in x]
    ess_res = [round(xi / sum(ess), 4) for xi in ess]
    return ess_res


# Used in Flash Attention
def softmax_online(x, blocks=2):
    if not isinstance(x, list):
        raise Exception("Please send a list for softmax")
    jump_size = len(x) // blocks
    mc, l, mc_old = -100, None, -100
    for i in range(0, len(x), jump_size):
        mid_arr = x[i : i + jump_size]
        print(f"{mid_arr=}")
        mc = max(mc, max(mid_arr))
        if l is not None and mc_old != mc:
            l = l * math.exp(mc_old - mc)
        mc_old = mc
        print(f"{mc=}")
        ess = [math.exp(xi - mc) for xi in mid_arr]
        if not l:
            l = sum(ess)
        else:
            l = l + sum(ess)

    ess_res = [math.exp(xi - mc) / l for xi in x]
    print(f"{ess_res=}")

In [29]:
l = [1.0, 3.0, 5.0, 6.0, 7.0, 2.0]
softmax_(l), torch.softmax(torch.tensor(l), dim=0), softmax_online(l)

mid_arr=[1.0, 3.0, 5.0]
mc=5.0
mid_arr=[6.0, 7.0, 2.0]
mc=7.0
ess_res=[0.0016193087920499101, 0.01196516350594841, 0.08841126437833055, 0.24032673339070443, 0.6532757922688734, 0.004401737664093237]


([0.0016, 0.012, 0.0884, 0.2403, 0.6533, 0.0044],
 tensor([0.00, 0.01, 0.09, 0.24, 0.65, 0.00]),
 None)

### Adding O

In [38]:
# Used in Flash Attention
def softmax_online_with_o(x, v, blocks=2):
    jump_size = len(x) // blocks
    mc, l, o = -100, None, None
    for i in range(0, len(x), jump_size):
        mid_arr = x[i : i + jump_size]
        mid_v = v[i : i + jump_size]
        new_mc = max(mc, max(mid_arr))

        # Rescale previous accumulated values
        if l is not None:
            scale = math.exp(mc - new_mc)
            o = o * scale
            l = l * scale
        mc = new_mc

        exp_values = [math.exp(xi - mc) for xi in mid_arr]
        block_l = sum(exp_values)
        block_o = sum(ei * vi for ei, vi in zip(exp_values, mid_v))

        if l is None:
            l = block_l
            o = block_o
        else:
            l += block_l
            o += block_o

    return o / l

In [48]:
l = [1.0, 3.0, 5.0, 6.0, 7.0, 2.0]
l_t = torch.tensor(l)
V = torch.tensor([1.1, 2.3, 1.4, 1.2, 1.3, 2.7])
torch.tensor(softmax_(l)) @ V, torch.softmax(l_t, dim=0) @ V, softmax_online_with_o(
    l_t, V
)

(tensor(1.30), tensor(1.30), tensor(1.30))

In [49]:
blocks = 2
step_size = l_t.shape[-1] // blocks
iter, mc, l, o = 1, -100, None, None
for i in range(0, l_t.shape[-1], step_size):
    # print(f"\nIter {iter}:")
    partial_l = l_t[i : i + step_size]
    partial_v = V[i : i + step_size]
    # print(f"{partial_l=} {partial_v=}")
    mc_new = max(mc, torch.max(partial_l))
    # print(f"{mc_new=}")
    if l is not None:
        scale = math.exp(mc - mc_new)
        l = l * scale
        o = o * scale
        # print(f"{scale=} {l=} {o=}")

    mc = mc_new

    exp_scales = [math.exp(xi - mc_new) for xi, vi in zip(partial_l, partial_v)]
    l_current = sum(exp_scales)
    o_current = sum([exp_i * vi for exp_i, vi in zip(exp_scales, partial_v)])
    # print(f"{exp_scales=} {l=}")

    if l is None:
        l, o = l_current, o_current
    else:
        l, o = l + l_current, o + o_current

    iter += 1

print(f"{(o/l)=}")

(o/l)=tensor(1.30)
